# Spark 4 Cluster - Notebook de verificacion

Comprueba la conexion con todos los servicios: Spark 4, Hive/MySQL, MinIO (S3)
Incluye ejemplos de novedades de Spark 4: VARIANT, SQL UDFs, pipe syntax.

## Crear SparkSession con Hive + S3

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://iabd-spark-master:7077")
    .appName("spark4-full-stack-demo")
    # --- Hive Metastore en MySQL (config explicita) ---
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.sql.warehouse.dir", "s3a://warehouse/hive")
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:mysql://iabd-mysql:3306/hive_metastore?useSSL=false&allowPublicKeyRetrieval=true")
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "com.mysql.cj.jdbc.Driver")
    .config("spark.hadoop.javax.jdo.option.ConnectionUserName", "hive")
    .config("spark.hadoop.javax.jdo.option.ConnectionPassword", "hivepass")
    .config("spark.hadoop.hive.metastore.schema.verification", "false")
    .config("spark.hadoop.hive.metastore.schema.verification.record.version", "false")
    .config("spark.hadoop.datanucleus.schema.autoCreateAll", "false")
    .config("spark.hadoop.datanucleus.autoCreateSchema", "false")
    .config("spark.hadoop.datanucleus.fixedDatastore", "true")
    .config("spark.hadoop.datanucleus.schema.autoCreateTables", "false")
    .config("spark.hadoop.datanucleus.schema.validateTables", "false")
    .config("spark.hadoop.datanucleus.schema.validateConstraints", "false")
    .config("spark.hadoop.datanucleus.schema.validateColumns", "false")
    # --- JDBC driver + extra jars (driver y executors) ---
    .config("spark.driver.extraClassPath", "/opt/spark/extra-jars/*:/opt/spark/jars/*")
    .config("spark.executor.extraClassPath", "/opt/spark/extra-jars/*:/opt/spark/jars/*")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.memory", "4g")
    .config("spark.network.timeout", "300s")
    .config("spark.executor.heartbeatInterval", "60s")
        
    # --- MinIO / S3A ---
    .config("spark.hadoop.fs.s3a.endpoint", "http://iabd-minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        
    # --- Paquetes (Kafka) ---
    # .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0")
        
    # --- ANSI mode (por defecto en Spark 4) ---
    .config("spark.sql.ansi.enabled", "true")
    .enableHiveSupport()
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Spark UI (driver): http://localhost:4040")
print(f"Spark Master UI:   http://localhost:8080")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/12 17:45:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Spark UI (driver): http://localhost:4040
Spark Master UI:   http://localhost:8080


In [2]:
spark

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Hive + MySQL Metastore

In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS demo")
spark.sql("USE demo")

26/04/12 17:45:45 WARN ObjectStore: Failed to get database demo, returning NoSuchObjectException
26/04/12 17:45:45 WARN ObjectStore: Failed to get database demo, returning NoSuchObjectException
26/04/12 17:45:45 WARN ObjectStore: Failed to get database demo, returning NoSuchObjectException


DataFrame[]

In [4]:
spark.sql("SELECT 1 as id, 'test' as nombre").show()

+---+------+
| id|nombre|
+---+------+
|  1|  test|
+---+------+



In [5]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
|     demo|
+---------+



In [6]:
spark.sql("USE demo")
print("USE demo OK")

USE demo OK


In [7]:
spark.sql("CREATE TABLE IF NOT EXISTS demo.test_local (id INT, nombre STRING) USING parquet")
print("Table creada OK")

26/04/12 17:45:49 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/04/12 17:45:49 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


Table creada OK


In [8]:
spark.sql("INSERT INTO demo.test_local VALUES (1, 'hola')")
spark.sql("SELECT * FROM demo.test_local").show()

+---+------+
| id|nombre|
+---+------+
|  1|  hola|
+---+------+



In [9]:
spark.sql("USE demo")
spark.sql("""
    CREATE TABLE IF NOT EXISTS ventas (
        id INT,
        producto STRING,
        cantidad INT,
        precio DOUBLE,
        fecha DATE
    )
    COMMENT 'Tabla de ejemplo de ventas'
""")

DataFrame[]

In [10]:
spark.sql("SELECT * FROM ventas ORDER BY fecha").show()

+---+--------+--------+------+-----+
| id|producto|cantidad|precio|fecha|
+---+--------+--------+------+-----+
+---+--------+--------+------+-----+



In [11]:
spark.sql("""
    INSERT INTO ventas VALUES
    (1, 'Laptop',   5,  999.99, DATE '2025-01-15'),
    (2, 'Mouse',    50, 19.99,  DATE '2025-01-16'),
    (3, 'Teclado',  30, 49.99,  DATE '2025-01-17'),
    (4, 'Monitor',  10, 349.99, DATE '2025-02-01'),
    (5, 'Webcam',   20, 79.99,  DATE '2025-02-05')
""")

DataFrame[]

In [12]:
spark.sql("SELECT * FROM ventas ORDER BY fecha").show()
spark.sql("SHOW TABLES IN demo").show()

+---+--------+--------+------+----------+
| id|producto|cantidad|precio|     fecha|
+---+--------+--------+------+----------+
|  1|  Laptop|       5|999.99|2025-01-15|
|  2|   Mouse|      50| 19.99|2025-01-16|
|  3| Teclado|      30| 49.99|2025-01-17|
|  4| Monitor|      10|349.99|2025-02-01|
|  5|  Webcam|      20| 79.99|2025-02-05|
+---+--------+--------+------+----------+

+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|     demo|test_local|      false|
|     demo|    ventas|      false|
+---------+----------+-----------+



## MinIO (S3) - Leer y escribir Parquet

In [13]:
df = spark.sql("SELECT * FROM demo.ventas")
df.write.mode("overwrite").parquet("s3a://processed/ventas_parquet")

df_s3 = spark.read.parquet("s3a://processed/ventas_parquet")
df_s3.show()
print(f"Registros leidos desde MinIO: {df_s3.count()}")

+---+--------+--------+------+----------+
| id|producto|cantidad|precio|     fecha|
+---+--------+--------+------+----------+
|  4| Monitor|      10|349.99|2025-02-01|
|  5|  Webcam|      20| 79.99|2025-02-05|
|  3| Teclado|      30| 49.99|2025-01-17|
|  1|  Laptop|       5|999.99|2025-01-15|
|  2|   Mouse|      50| 19.99|2025-01-16|
+---+--------+--------+------+----------+

Registros leidos desde MinIO: 5


### Resumen del entorno

In [14]:

print("=" * 60)
print(" RESUMEN DEL ENTORNO - SPARK 4")
print("=" * 60)
print(f" Spark Master UI:   http://localhost:8080")
print(f" Spark Driver UI:   http://localhost:4040")
print(f" Jupyter Lab:       http://localhost:8888 (token: spark)")
print(f" MinIO Console:     http://localhost:9001 (minioadmin / minioadmin123)")
print(f" Kafka UI:          http://localhost:8081")
print(f" MySQL:             localhost:3306 (hive / hivepass)")
print(f"")
print(f" Spark version:     {spark.version}")
print(f" ANSI mode:         {spark.conf.get('spark.sql.ansi.enabled')}")
print(f" Workers activos:   {spark.sparkContext.defaultParallelism}")
print("=" * 60)

 RESUMEN DEL ENTORNO - SPARK 4
 Spark Master UI:   http://localhost:8080
 Spark Driver UI:   http://localhost:4040
 Jupyter Lab:       http://localhost:8888 (token: spark)
 MinIO Console:     http://localhost:9001 (minioadmin / minioadmin123)
 Kafka UI:          http://localhost:8081
 MySQL:             localhost:3306 (hive / hivepass)

 Spark version:     4.1.1
 ANSI mode:         true
 Workers activos:   4
